In [1]:
import sys
print(sys.executable)

/workspaces/21-days-with-AI-agent/llm-zoomcamp-code/.llmvenv/bin/python


In [1]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from google import genai

google_client = genai.Client()

documents = load_faq_data()
index = build_index(documents)



assistant = RAGBase(
    index=index,
    llm_client=google_client,
)



In [3]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning whenever you want, as the videos and GitHub materials are available.


In [3]:
assistant.rag("How do I get a certificate?")


'To get a certificate, you need to:\n\n1.  Finish the course with a "live" cohort. Certificates are not awarded for the self-paced mode.\n2.  Pass the Capstone project.\n3.  Peer-review 3 capstones after submitting your project, which can only be done while the course is running.\n4.  Ensure your official name, as it appears on your identification documents, is entered in the second field of your course profile settings, as this is the name that will appear on your certificate.'

In [4]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course even after it has started. The videos and GitHub materials are available for you to start whenever you want.\n\nHowever, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Also, remember that you can only get a certificate if you finish the course with a "live" cohort, as you need to peer-review projects while the course is running.'

In [4]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 85 documents


In [7]:
import time
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

for doc in docs_llm:
    sqlite_index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why are we not using Langchain in the course?...
Added: OpenAI: Error when running OpenAI respon

In [8]:
assistant = RAGBase(
    index=sqlite_index,
    llm_client=google_client,
)

In [ ]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Comparing the two approaches
With minsearch (single process):

Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
With sqlitesearch (two processes):

Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready

flowchart TD

    subgraph RAG["RAG ASSISTANT"]
        U([🙂 User])
        APP[Application]
        DOCS[[D1 ... D5]]
        PROMPT[Build Prompt<br/>Question + Context]
        LLM[LLM]
        ANSWER([Answer])

        U -->|Question| APP
        DOCS --> APP
        APP --> PROMPT
        PROMPT --> LLM
        LLM --> ANSWER
        ANSWER --> U
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS


In [ ]:
# When you're done, close the database connection:
sqlite_index.close()
# Or just let Python clean it up when the notebook kernel shuts down.

# Defining the tool

### First method

“There is a function called search.”

“It takes one argument: query.”

“Use it when useful.”

But Gemini still does not have your Python implementation of search function

So after Gemini returns a function call, your code must do the rest:

read the function name and args,

call the real Python function,

send the function result back

In [5]:
from google import genai
from google.genai import types

search_fn = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters={
        "type": "OBJECT",
        "properties": {
            "query": {
                "type": "STRING",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"]
    }
)

tool = types.Tool(function_declarations=[search_fn])

response = google_client.models.generate_content(
    model="gemini-2.5-flash",
    contents="I just discovered the course. Can I join it?",
    config=types.GenerateContentConfig(tools=[tool])
)

print(response.text)

None


In [ ]:
import json
from google.genai import types

# The function call contains JSON arguments. 
# We parse them, call our search function, and serialize the result.


# Get the first function call from Gemini
call = response.function_calls[0]
print(call)
# response like above is basically saying we need to call search function 
# and the input query is "join course" 

# Args are already structured in Gemini
args = call.args

print(args)

# Execute your real Python function: search
results = assistant.search(**args)
print(results)

# Optional: serialize if you want to inspect/debug
result_json = json.dumps(results, indent=2)

id=None args={'query': 'join course'} name='search' partial_args=None will_continue=None
{'query': 'join course'}
[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': 'bd31146b0e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'When will the course be offered next?', 'answer': 'Summer 2027.'}, {'id': '04919992b3', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How should I start the course and follow the weekly workflow?', 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github

In [10]:

# Now we send this result back to the model. 
# First, we add the model's output to the conversation history 
# - the model needs to see its own function call. Then we add the tool result.
contents = [
    types.Content(role="user", parts=[
        types.Part(text="I just discovered the course. Can I join it?")
    ]),
    types.Content(role="model", parts=[
        types.Part(function_call=call)
    ]),
    types.Content(role="user", parts=[
        types.Part(function_response=types.FunctionResponse(
            name=call.name,
            response={"result": results}
        ))
    ]),
]

In [11]:
# Asking the model again
response = google_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=contents,
    config=types.GenerateContentConfig(tools=[tool])
)

print(response.text)

Yes, you can join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available. The deadlines are listed in the course management platform. Note that you can only get a certificate if you finish the course with a "live" cohort, as it involves peer-reviewing projects, which can only be done while the course is running.


### Second method
the SDK already has the real Python function. So if automatic function calling is enabled, it can:

derive the tool definition from the function signature and docstring,

execute the function,

pass the result back to the model,

repeat if needed,

return the final natural-language answer.

In [ ]:
# Since you already have RAGBase.search, 
# the easiest pattern is to create a tiny wrapper function outside the class that calls your class method
def search(query: str):
    """Search the course FAQ for relevant entries.

    Args:
        query: Search query text to look up in the llm-zoomcamp FAQ.
    """
    return assistant.search(query)

In [ ]:
# Sending the question with the tool

response = google_client.models.generate_content(
    model="gemini-2.5-flash",
    contents="I just discovered the course. Can I join it?",
    config=types.GenerateContentConfig(
        tools=[search]
    )
)

print(response.text)

Yes, you can join the course. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.


# The Agentic Loop
In the previous lesson, we did function calling by hand. We sent a message and got back a function call. We ran it, sent the result back, and got the answer.

That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. An agent is exactly that.

#### this part is not necessary if we already have method 2 above. but still important to understand and sometimes necessary for advanced control.